In [1]:
import sys, os

In [2]:
# notebook parameters, will be replaced by script arguments
application_name = "My Application"
s3a_access_key = os.environ.get("s3a_access_key")
s3a_secret_key = os.environ.get("s3a_secret_key")
input_file_1 = "s3a://uga-data-lake/sf-loan-performance-data-sample.csv"
output_file_1 = "s3a://uga-data-lake/sf-loan-performance-data-out.csv"

In [3]:
import pyspark
from pyspark import SparkContext, SparkConf, SQLContext
from pyspark.sql import SparkSession
import pyspark.sql.types as T
import pyspark.sql.functions as F
pyspark.__version__

'4.1.2'

In [4]:
# test if the script is running as a standalone program or a notebook
if __name__ == '__main__' and '__file__' in globals():
    print("Running as a script")
    application_name = sys.argv[1]
    s3a_access_key = sys.argv[2]
    s3a_secret_key = sys.argv[3]
    input_file_1 = sys.argv[4]
    output_file_1 = sys.argv[5]
    interactive = False
else:
    interactive = True
    print("Running as a notebook")
    pv = !{sys.executable} --version
    print("Python version ", pv)
    os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
    os.environ.setdefault("PYSPARK_DRIVER_PYTHON", sys.executable)
    print("PySpark version ", pyspark.__version__)
    jv = !java --version
    print("Java version ", jv)

Running as a notebook
Python version  ['Python 3.12.13']
PySpark version  4.1.2
Java version  ['openjdk 17.0.19 2026-04-21', 'OpenJDK Runtime Environment (build 17.0.19+10-1-deb12u2-Debian)', 'OpenJDK 64-Bit Server VM (build 17.0.19+10-1-deb12u2-Debian, mixed mode, sharing)']


In [5]:
def show(df):
    if interactive:
        df.show()

In [6]:
spark = (SparkSession.builder.appName(application_name)
         .config('spark.jars.packages', 'org.apache.hadoop:hadoop-aws:3.5.0')
         .config('spark.hadoop.fs.s3a.access.key', s3a_access_key)
         .config("spark.hadoop.fs.s3a.secret.key", s3a_secret_key)
         .config("spark.hadoop.fs.s3a.endpoint", "s3.us-east-2.amazonaws.com")
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("ERROR")
spark

:: loading settings :: url = jar:file:/usr/local/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-04cdc6c2-3c92-439f-9834-3c99d4758f75;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.5.0 in central
	found software.amazon.awssdk#bundle;2.35.4 in central
	found software.amazon.s3.analyticsaccelerator#analyticsaccelerator-s3;1.3.1 in central
	found org.wildfly.openssl#wildfly-openssl;2.2.5.Final in central
:: resolution report :: resolve 111ms :: artifacts dl 4ms
	:: modules in use:
	org.apache.hadoop#hadoop-aws;3.5.0 from central in [default]
	org.wildfly.openssl#wildfly-openssl;2.2.5.Final from central in [default]
	software.amazon.awssdk#bundle;2.35.4 from central in [default]
	software.amazon.s3.analyt

The file sf-loan-performance-data-sample.csv contains a sample from Fannie Mae loan performance data https://capitalmarkets.fanniemae.com/credit-risk-transfer/single-family-credit-risk-transfer/fannie-mae-single-family-loan-performance-data

The data dictionary is provided here https://capitalmarkets.fanniemae.com/media/6931/display

Our data scientists are interested in columns

- 2 Loan Identifier
- 20 Original Loan to Value Ratio (LTV)
- 31 Property State
- 33 Zip Code Short
- extract these columns, and select only loans in FL        
- save the results in csv format

This drill packages the Fannie Mae transformation from `spark-dataframe-drill.ipynb`  
Convert it into a pipeline-style notebook, mirroring what would run via `spark-submit` on local cluster and AWS EMR.



In [7]:
data = ( spark.read.option("header", "false")
            .option("multiLine", "false")
            .option("inferSchema", "false")
            .option("delimiter", "|")
            .csv(input_file_1) )
show(data)

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


+----+------------+------+---+-----+-----+----+-----+-----+--------+----+--------+----+------+------+----+----+----+------+----+----+----+----+----+----+----+----+----+----+----+----+-----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+-----+-----+-----+-----+-----+-----+-----+-----+
| _c0|         _c1|   _c2|_c3|  _c4|  _c5| _c6|  _c7|  _c8|     _c9|_c10|    _c11|_c12|  _c13|  _c14|_c15|_c16|_c17|  _c18|_c19|_c20|_c21|_c22|_c23|_c24|_c25|_c26|_c27|_c28|_c29|_c30| _c31|_c32|_c33|_c34|_c35|_c36|_c37|_c38|_c39|_c40|_c41|_c42|_c43|_c44|_c45|_c46|_c47|_c48|_c49|_c50|_c51|_c52|_c53|_c54|_c55|_c56|_c57|_c58|_c59|_c60|_c61|_c62|_c63|_c64|_c65|_c66|_c67|_c68|_c69|_c70|_c71|_c72|_c73|_c74|_c75|_c76|_c77|_c7

In [8]:
step1 = data.select(F.col("_c1"), F.col("_c19"), F.col("_c30"), F.col("_c32"))
step2 = step1.where(F.col("_c30") == "FL")
step2.write.mode("overwrite").option("header", "false").csv(output_file_1)

In [9]:
spark.stop()